In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt

# Import necessary dataset modules
from dataset.data_loader.InfantDataLoader import InfantDataLoader

# ==========================================
# 1. ARCHITECTURAL / MODEL REGISTRY
# ==========================================
# Import your candidate models here
from neural_methods.model.iBVPNet import iBVPNet
# from neural_methods.model.PhysNet import PhysNet  # Example for other models

# --- CHANGE THESE TO SWITCH MODELS ---
MODEL_CLASS = iBVPNet  
MODEL_PATH = "./kfold_outputs/iBVPNet_Fold1_BestLoss.pth"

# Model Parameters
MODEL_SPECS = {
    "frames": 150,
    "in_channels": 1,
    "data_format": "NCDHW",       # "NCDHW" (e.g., iBVPNet, PhysNet) or "NDCHW" (e.g., TS-CAN)
    "requires_ibvp_padding": True # True ONLY for iBVPNet; set False for standard models
}

# ==========================================
# 2. CONFIGURATION & DATASET LOADING
# ==========================================
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

class ConfigMock:
    def __init__(self, dictionary):
        for key, value in dictionary.items():
            if isinstance(value, dict):
                setattr(self, key, ConfigMock(value))
            else:
                setattr(self, key, value)
OUTPUT_DIR = "./kfold_outputs"
# Set up loader config, matching the target model's data format orientation
loader_config_dict = {
    'CACHED_PATH': 'C:/Users/HD/Documents/infant_ecg_analysis/preprocessed_data/preprocessed_cache_72x_150f_d-raw_l-raw',
    'FILE_LIST_PATH': 'C:/Users/HD/Documents/infant_ecg_analysis/preprocessed_data/preprocessed_cache_72x_150f_d-raw_l-raw.csv',
    'BEGIN': 0.0,          
    'END': 1.0,
    'DATA_FORMAT': 'NCDHW',
    'DO_PREPROCESS': False,
    'VIDEO_FPS': 30.0,
    'SIGNAL_FS': 500.0,
    'STATS_CSV': {'AVAILABLE': True, 'PATH': 'C:/Users/HD/Documents/infant_ecg_analysis/preprocessed_data/preprocessed_cache_72x_150f_d-raw_l-raw_participant_stats.csv'},
    'PREPROCESS': {'DATA_TYPE': ['Raw'], 'LABEL_TYPE': 'Standardized'},
    'TEST': {'DATA': {'FS': 30.0}, 'METRICS': ['MAE', 'RMSE', 'MAPE', 'Pearson', 'SNR']},
    'INFERENCE': {'EVALUATION_METHOD': 'FFT', 'SAVE_PATH': OUTPUT_DIR, 'EVALUATION_WINDOW': {'USE_SMALLER_WINDOW': False}}
}

config_data = ConfigMock(loader_config_dict)

print("Loading dataset...")
dataset = InfantDataLoader(
    dataset_name="Infant_Dataset",
    raw_data_path="G:/image/relax",
    label_data_path="G:/pulse_data/relax",
    config_data=config_data,
    device=DEVICE
)

# ==========================================
# 3. DYNAMIC MODEL INITIALIZATION
# ==========================================
print(f"Initializing {MODEL_CLASS.__name__}...")
# Instantiate safely passing dimensions regardless of unique naming rules
if MODEL_CLASS.__name__ == "iBVPNet":
    model = MODEL_CLASS(frames=MODEL_SPECS["frames"], in_channels=MODEL_SPECS["in_channels"]).to(DEVICE)
else:
    # Standard architectures often use 'num_frames' or default positioning args
    model = MODEL_CLASS().to(DEVICE) 

print(f"Loading checkpoint weights from {MODEL_PATH}...")
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

# ==========================================
# 4. SAMPLE SELECTION AND INFERENCE ADAPTATION
# ==========================================
sample_index = 300
data, label, filename, chunk_id = dataset[sample_index]

# Add Batch Dimension -> [1, ...]
data_tensor = torch.from_numpy(data).unsqueeze(0).to(DEVICE)

# Apply model-specific preprocessing adapters conditionally
if MODEL_SPECS["requires_ibvp_padding"]:
    # Temporal dimension is at index 2 for NCDHW format
    last_frame = torch.unsqueeze(data_tensor[:, :, -1, :, :], 2)
    data_tensor = torch.cat((data_tensor, last_frame), 2)

print(f"Running inference with {MODEL_CLASS.__name__} on sample {sample_index}...")
with torch.no_grad():
    prediction = model(data_tensor)
    prediction = prediction.squeeze(0).cpu().numpy()

# Flatten or extract if the model outputs extra dimensions (e.g., [1, T] vs [T])
prediction = prediction.flatten()

# ==========================================
# 5. STANDARDIZATION & PLOTTING
# ==========================================
prediction_normalized = (prediction - np.mean(prediction)) / (np.std(prediction) + 1e-7)
label_normalized = (label - np.mean(label)) / (np.std(label) + 1e-7)

# Handle cases where model sequence outputs differ slightly from ground truth lengths
min_len = min(len(prediction_normalized), len(label_normalized))
time_axis = np.arange(min_len)

plt.figure(figsize=(12, 5))
plt.plot(time_axis, label_normalized[:min_len], label='True Label (BVP GT)', color='black', alpha=0.7, linewidth=2)
plt.plot(time_axis, prediction_normalized[:min_len], label=f'Predicted Wave ({MODEL_CLASS.__name__})', color='crimson', linestyle='--', linewidth=2)

plt.title(f"Model Inference: {filename} (Chunk {chunk_id}) | Model: {MODEL_CLASS.__name__}", fontsize=13)
plt.xlabel("Frames", fontsize=11)
plt.ylabel("Normalized Amplitude", fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()